# TP04: Dashboards Interactivos
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 2: Herramientas en la Nube de Visualización de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Diseñar **activos visuales dinámicos** para monitoreo de negocios
2. Crear **gráficos interactivos** con Plotly Express
3. Agrupar visualizaciones en un **panel consolidado**
4. Desarrollar **dashboards funcionales** integrados en el notebook

---

### 📁 Caso de Estudio: Dashboard de Ventas de Panadería

Crearemos un dashboard ejecutivo para la Panadería La Espiga Dorada con:
* Indicadores clave (KPIs)
* Gráficos de tendencias
* Análisis por sucursal y producto
* Visualizaciones interactivas

### 🕰️ Duración Estimada: 3 horas

In [0]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configurar estilo
sns.set_palette('husl')
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Librerías importadas")
print("🎨 Listo para crear dashboards")

## Parte 1: Carga y Preparación de Datos

### 📂 Cargar datos consolidados

Vamos a cargar y preparar los datos para el dashboard ejecutivo.

In [0]:
# Cargar datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_productos = pd.read_csv(ruta_datos + 'productos.csv')
df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')
df_ventas = pd.read_csv(ruta_datos + 'ventas.csv', parse_dates=['fecha'])
df_detalles = pd.read_csv(ruta_datos + 'detalles_ventas.csv')

# Crear dataset consolidado
df_dash = df_detalles.merge(df_productos[['producto_id', 'nombre', 'categoria']], on='producto_id').merge(
    df_ventas[['venta_id', 'fecha', 'sucursal_id']], on='venta_id'
).merge(
    df_sucursales[['sucursal_id', 'nombre', 'zona']], on='sucursal_id', suffixes=('_producto', '_sucursal')
)

df_dash['mes'] = df_dash['fecha'].dt.to_period('M')
df_dash['dia_semana'] = df_dash['fecha'].dt.day_name()

print(f"✅ Dataset consolidado: {len(df_dash):,} registros")
display(df_dash.head())

## Parte 2: Indicadores Clave de Desempeño (KPIs)

### 📊 Métricas ejecutivas principales

In [0]:
# Calcular KPIs principales
facturacion_total = df_dash['subtotal'].sum()
total_transacciones = df_dash['venta_id'].nunique()
ticket_promedio = facturacion_total / total_transacciones
productos_vendidos = df_dash['cantidad'].sum()

print("📊 INDICADORES CLAVE DE DESEMPEÑO (KPIs)")
print("=" * 80)
print(f"\n💰 Facturación Total:        ${facturacion_total:>20,.2f}")
print(f"📝 Total Transacciones:    {total_transacciones:>20,}")
print(f"🎯 Ticket Promedio:        ${ticket_promedio:>20,.2f}")
print(f"📦 Productos Vendidos:     {int(productos_vendidos):>20,}")

print("\n" + "=" * 80)

## Parte 3: Dashboard Visual Integrado

### 📈 Panel de visualizaciones ejecutivas

Crearemos un conjunto de gráficos que forman un dashboard completo.

In [0]:
# 1. Tendencia de ventas mensuales
ventas_mes = df_dash.groupby('mes').agg({
    'subtotal': 'sum',
    'venta_id': 'nunique'
}).reset_index()
ventas_mes.columns = ['mes', 'facturacion', 'transacciones']
ventas_mes['mes_str'] = ventas_mes['mes'].astype(str)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Facturación mensual
ax1.plot(ventas_mes['mes_str'], ventas_mes['facturacion'], 
         marker='o', linewidth=2.5, markersize=8, color='steelblue')
ax1.fill_between(range(len(ventas_mes)), ventas_mes['facturacion'], alpha=0.3, color='steelblue')
ax1.set_title('💰 Evolución de Facturación Mensual', fontsize=16, fontweight='bold', pad=20)
ax1.set_xlabel('Mes', fontsize=12)
ax1.set_ylabel('Facturación ($)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Transacciones mensuales
ax2.bar(ventas_mes['mes_str'], ventas_mes['transacciones'], color='coral', edgecolor='black', alpha=0.7)
ax2.set_title('📝 Número de Transacciones Mensuales', fontsize=16, fontweight='bold', pad=20)
ax2.set_xlabel('Mes', fontsize=12)
ax2.set_ylabel('Transacciones', fontsize=12)
ax2.grid(axis='y', alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [0]:
# 2. Rendimiento por sucursal
ventas_sucursal = df_dash.groupby('nombre_sucursal').agg({
    'subtotal': 'sum',
    'venta_id': 'nunique',
    'cantidad': 'sum'
}).reset_index()
ventas_sucursal.columns = ['sucursal', 'facturacion', 'transacciones', 'unidades']
ventas_sucursal = ventas_sucursal.sort_values('facturacion', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Facturación por sucursal
axes[0].barh(ventas_sucursal['sucursal'], ventas_sucursal['facturacion'], color='teal', edgecolor='black')
axes[0].set_title('🏢 Facturación por Sucursal', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Facturación ($)')
for i, v in enumerate(ventas_sucursal['facturacion']):
    axes[0].text(v, i, f' ${v/1e6:.1f}M', va='center', fontsize=10)

# Transacciones por sucursal
axes[1].barh(ventas_sucursal['sucursal'], ventas_sucursal['transacciones'], color='orange', edgecolor='black')
axes[1].set_title('📝 Transacciones por Sucursal', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Transacciones')
for i, v in enumerate(ventas_sucursal['transacciones']):
    axes[1].text(v, i, f' {int(v):,}', va='center', fontsize=10)

# Unidades vendidas por sucursal
axes[2].barh(ventas_sucursal['sucursal'], ventas_sucursal['unidades'], color='purple', edgecolor='black')
axes[2].set_title('📦 Unidades Vendidas', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Unidades')
for i, v in enumerate(ventas_sucursal['unidades']):
    axes[2].text(v, i, f' {int(v):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [0]:
# 3. Top productos más vendidos
top_productos = df_dash.groupby(['nombre_producto', 'categoria']).agg({
    'cantidad': 'sum',
    'subtotal': 'sum'
}).reset_index().sort_values('subtotal', ascending=False).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top por facturación
y_pos = np.arange(len(top_productos))
colors = plt.cm.Set3(np.linspace(0, 1, len(top_productos)))
ax1.barh(y_pos, top_productos['subtotal'], color=colors, edgecolor='black')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_productos['nombre_producto'])
ax1.set_xlabel('Facturación ($)', fontsize=12)
ax1.set_title('🏆 Top 10 Productos por Facturación', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(top_productos['subtotal']):
    ax1.text(v, i, f' ${v/1e6:.2f}M', va='center', fontsize=9)

# Top por unidades
top_unidades = df_dash.groupby(['nombre_producto', 'categoria']).agg({
    'cantidad': 'sum'
}).reset_index().sort_values('cantidad', ascending=False).head(10)

y_pos2 = np.arange(len(top_unidades))
colors2 = plt.cm.Pastel1(np.linspace(0, 1, len(top_unidades)))
ax2.barh(y_pos2, top_unidades['cantidad'], color=colors2, edgecolor='black')
ax2.set_yticks(y_pos2)
ax2.set_yticklabels(top_unidades['nombre_producto'])
ax2.set_xlabel('Unidades Vendidas', fontsize=12)
ax2.set_title('📦 Top 10 Productos por Unidades', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
for i, v in enumerate(top_unidades['cantidad']):
    ax2.text(v, i, f' {int(v):,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [0]:
# 4. Análisis por categoría y día de semana
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Ventas por categoría (pie chart)
ventas_cat = df_dash.groupby('categoria')['subtotal'].sum().sort_values(ascending=False)
colors_pie = plt.cm.Set2(np.linspace(0, 1, len(ventas_cat)))
wedges, texts, autotexts = ax1.pie(ventas_cat, labels=ventas_cat.index, autopct='%1.1f%%',
                                     colors=colors_pie, startangle=90, textprops={'fontsize': 10})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax1.set_title('🎨 Distribución de Facturación por Categoría', fontsize=14, fontweight='bold')

# Ventas por día de semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ventas_dia = df_dash.groupby('dia_semana')['subtotal'].sum().reindex(orden_dias)
colors_bar = ['#FF6B6B' if i >= 5 else '#4ECDC4' for i in range(7)]
ax2.bar(range(7), ventas_dia, color=colors_bar, edgecolor='black', alpha=0.8)
ax2.set_xticks(range(7))
ax2.set_xticklabels(['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom'])
ax2.set_title('📅 Facturación por Día de Semana', fontsize=14, fontweight='bold')
ax2.set_ylabel('Facturación ($)')
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

## 🎯 Resumen del TP04

### ✅ Qué aprendimos:

1. **Cálculo de KPIs**: Métricas ejecutivas clave para el negocio
2. **Visualizaciones múltiples**: Gráficos de líneas, barras, pie charts
3. **Dashboard integrado**: Panel consolidado con múltiples vistas
4. **Análisis temporal**: Tendencias mensuales y semanales
5. **Comparación de sucursales**: Rendimiento relativo
6. **Top productos**: Identificación de mejores performers

### 💡 Insights del dashboard:

* Hay variación significativa en ventas por mes
* Los fines de semana tienen mayor facturación
* Ciertas categorías dominan las ventas
* Las sucursales tienen rendimientos diferenciados

### 🚀 Próximos pasos:

En la **Unidad 3** (Modelado de Datos) aprenderemos a:
* Crear estructuras de datos optimizadas
* Aplicar agregaciones complejas
* Diseñar modelos de datos eficientes
* Preparar datos para machine learning

---

**📝 Excelente! Has creado un dashboard ejecutivo completo para la toma de decisiones.**

### 📚 UNIDAD 1 Y 2 COMPLETADAS ✅

Has finalizado las primeras dos unidades del curso:
* **Unidad 1 - Análisis de Datos**: Configuración, carga, transformación y exploración
* **Unidad 2 - Visualización de Datos**: Perfilado y dashboards

Contínua con las **Unidades 3 y 4** para completar el programa del curso.